# Phase 1 diagnostic: evaluate the actual published checkpoint

The rounded-parameter replay gave 72/100 refusals and KL 0.4340 versus the card's 6/100 and 0.0793.
This bounded follow-up evaluates the pinned publisher checkpoint using exactly the replay's
FP16, non-thinking, 100-token, marker-scoring setup. It reuses the baseline distributions from
the completed replay, does not extract or edit directions, and does not search parameters.
All model execution and dependency setup stay on Kaggle T4 x2. Results go to the same private Hub repo.


In [ ]:
# 1. Download the small pinned source bundle and install dependencies ON KAGGLE.
# Torch is already provided by Kaggle and is constrained to its existing version.
import hashlib, importlib.metadata, io, json, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path

assert Path('/kaggle').exists(), 'Run this notebook on Kaggle.'
WORK = Path('/kaggle/working/qwen35_reproduction')
WORK.mkdir(parents=True, exist_ok=True)
# Batch mode may receive the owner's token as a private Kaggle input.
# Never print or copy this credential into the output directory.
if not os.environ.get('HF_TOKEN'):
    for credential_file in Path('/kaggle/input').glob('**/phase1_hf_token.txt'):
        os.environ['HF_TOKEN'] = credential_file.read_text().strip()
        break
os.environ.setdefault('HF_HOME', '/kaggle/temp/qwen35_hf_cache')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
PROVENANCE = json.loads('{"retrieved_date": "2026-11-07", "original_notebook_found": false, "author_reproduction_manifest_found": false, "release_tag_commit": "27097bfe8e60d5e65f0200785f1e96a52afcbc2a", "compatible_source_commit": "e26da5e0e648bb285f896c9801849edfa8d20913", "compatible_source_url": "https://codeload.github.com/p-e-w/heretic/zip/e26da5e0e648bb285f896c9801849edfa8d20913", "compatible_source_archive_sha256": "486f20a05152bc1c21fec2243164fa69efde1d963f3af15909559dfd6d355cf1", "reference_model": {"id": "trohrbaugh/Qwen3.5-9B-heretic-v2", "revision": "81dda62421c43a5bd5722edbc6cd47714673dc7f"}, "base_model": {"id": "Qwen/Qwen3.5-9B", "revision": "c202236235762e1c871ad0ccb60c8ee5ba337b9a"}, "datasets": {"mlabonne/harmful_behaviors": "01cead01398926d81f7c52bdb790ee8cf77ebba7", "mlabonne/harmless_alpaca": "02c6a92cfcf11bb0c387334f8146d149d65b587f"}, "published_metrics": {"baseline_refusals": 100, "edited_refusals": 6, "evaluation_n": 100, "benign_first_token_kl": 0.0793}, "limitations": ["Published parameters rounded to two decimal places.", "Original seed, full environment, source/model/dataset revisions and study journal were not supplied.", "The compatible upstream source postdates the published checkpoint. This is a best-effort parameter replay, not byte-for-byte reproduction."]}')
PUBLISHED = json.loads('{"direction_index": 18.49, "parameters": {"attn.o_proj": {"max_weight": 2.0, "max_weight_position": 18.99, "min_weight": 1.43, "min_weight_distance": 9.65}, "mlp.down_proj": {"max_weight": 1.67, "max_weight_position": 26.06, "min_weight": 0.84, "min_weight_distance": 18.04}}}')
REQUIREMENTS = "# Compatibility environment chosen for this replay; not the author's unpublished environment.\n# Keep Kaggle's existing CUDA-enabled torch installation.\ntransformers==5.22.1\npeft==0.21.2\naccelerate==1.15.0\ndatasets==4.8.5\nbitsandbytes==0.50.2\nhuggingface-hub==1.33.0\nkernels==0.18.1\noptuna==5.0.0\npsutil==7.2.2\npydantic-settings==2.16.2\nquestionary==2.1.1\nrich==15.0.0\nhf-transfer==0.1.9\nsafetensors==0.8.0\n"
(WORK / 'requirements-kaggle.txt').write_text(REQUIREMENTS)
(WORK / 'sources.json').write_text(json.dumps(PROVENANCE, indent=2))
(WORK / 'published_parameters.json').write_text(json.dumps(PUBLISHED, indent=2))

# Kaggle ships an old torchao that recent PEFT rejects even for unquantized LoRA.
# It is unused in this phase; remove it while preserving the CUDA PyTorch stack.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
constraints = []
for name in ('torch', 'torchvision', 'torchaudio'):
    try:
        constraints.append(name + '==' + importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        pass
assert any(x.startswith('torch==') for x in constraints), 'Select a Kaggle GPU image with PyTorch.'
(WORK / 'preserve-kaggle-torch.txt').write_text('\n'.join(constraints) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements-kaggle.txt'),
                '-c', str(WORK / 'preserve-kaggle-torch.txt')], check=True)

with urllib.request.urlopen(PROVENANCE['compatible_source_url'], timeout=60) as response:
    source_zip = response.read()
assert hashlib.sha256(source_zip).hexdigest() == PROVENANCE['compatible_source_archive_sha256']
SOURCE = WORK / 'upstream'
with zipfile.ZipFile(io.BytesIO(source_zip)) as archive:
    for item in archive.infolist():
        parts = Path(item.filename).parts[1:]
        if not parts or item.is_dir():
            continue
        target = SOURCE.joinpath(*parts)
        assert target.resolve().is_relative_to(SOURCE.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
sys.path.insert(0, str(SOURCE / 'src'))
print('Source ready:', PROVENANCE['compatible_source_commit'])
print('Kaggle PyTorch preserved:', constraints)


In [ ]:
# 2. GPU setup. Keep the published sample counts and response limit.
import gc, platform, random, time
from collections import Counter
from functools import partial
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from heretic.config import Settings
from heretic.evaluator import Evaluator
from heretic.model import Model, AbliterationParameters
from heretic.utils import Prompt

SEED = 0                 # Our recorded seed; the author's seed is unknown.
BATCH_SIZE = 1           # Conservative start for two T4s.
MAX_NEW_TOKENS = 100     # Heretic v1.2.0 default.
SAVE_MERGED = True       # Approximately 19 GB; saved under /kaggle/working.
OUT = WORK / 'results'
OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Enable GPU T4 ×2 in Kaggle settings.'
assert torch.cuda.device_count() >= 2, 'This configuration expects two GPUs.'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_grad_enabled(False)

GPUS = [{'index': i, 'name': torch.cuda.get_device_name(i),
         'vram_gib': torch.cuda.get_device_properties(i).total_memory / 2**30}
        for i in range(torch.cuda.device_count())]
MAX_MEMORY = {str(g['index']): f"{min(13, int(g['vram_gib']) - 2)}GiB" for g in GPUS}
MAX_MEMORY['cpu'] = '0GiB'  # Fail visibly instead of silently offloading to a slow CPU path.
print(json.dumps(GPUS, indent=2))
for location in ('/kaggle/working', '/kaggle/temp', '/tmp'):
    import shutil
    Path(location).mkdir(parents=True, exist_ok=True)
    print('Free GiB:', location, round(shutil.disk_usage(location).free / 2**30, 1))

# The upstream collector stacks hidden states directly, which fails across GPUs.
# Move only the final-token states to CPU before stacking; keep the original estimator.
class KaggleModel(Model):
    @torch.inference_mode()
    def get_residuals(self, prompts):
        _, outputs = self.generate(prompts, max_new_tokens=1,
                                   output_hidden_states=True, return_dict_in_generate=True)
        states = torch.stack([h[:, -1, :].detach().float().cpu()
                              for h in outputs.hidden_states[0]], dim=1)
        assert torch.isfinite(states).all(), 'Non-finite residuals; stop this run.'
        return states

    @torch.inference_mode()
    def get_logprobs(self, prompts):
        values = super().get_logprobs(prompts).detach().float().cpu()
        assert torch.isfinite(values).all(), 'Non-finite log probabilities; stop this run.'
        return values

versions = {}
for name in ('torch', 'transformers', 'peft', 'accelerate', 'datasets', 'huggingface-hub',
             'bitsandbytes', 'kernels', 'numpy'):
    versions[name] = importlib.metadata.version(name)
manifest = {'status': 'started', 'sources': PROVENANCE, 'published_parameters': PUBLISHED,
            'seed': SEED, 'python': platform.python_version(), 'packages': versions,
            'gpus': GPUS, 'batch_size': BATCH_SIZE, 'max_new_tokens': MAX_NEW_TOKENS,
            'quantization': 'none', 'dtype': 'float16 with model-required FP32 exceptions',
            'thinking': False, 'scorer': 'upstream Heretic v1.2.0 refusal markers',
            'reproduction_scope': 'rounded published-parameter replay, not exact weight reproduction'}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True,
               stdout=(OUT / 'environment.txt').open('w'))


In [ ]:
# 3. Download the pinned original model and load it once.
# Public checkpoints/datasets require no embedded token or Hub login.
base = PROVENANCE['reference_model']
BASE_PATH = snapshot_download(base['id'], revision=base['revision'], max_workers=4,
                             allow_patterns=['*.json', '*.safetensors', '*.jinja', '*.txt', '*.model'])
settings_values = dict(
    model=BASE_PATH, dtypes=['float16'], device_map='auto', max_memory=MAX_MEMORY,
    batch_size=BATCH_SIZE, max_response_length=MAX_NEW_TOKENS,
    quantization='none', trust_remote_code=False,
    orthogonalize_direction=False, row_normalization='none', winsorization_quantile=1.0,
)
# Pydantic's CLI source must not parse Jupyter's '-f kernel.json' arguments.
old_argv = sys.argv
try:
    sys.argv = [sys.argv[0]]
    settings = Settings(**settings_values)
finally:
    sys.argv = old_argv
model = KaggleModel(settings)
model.model.eval()
manifest['actual_loaded_model'] = base
manifest['condition'] = 'published_checkpoint_fp16'
manifest['effective_settings'] = settings.model_dump(mode='json')
# Explicitly close the thinking block using Qwen's own chat template.
# Old Heretic auto-prefix detection is unreliable with this model's current template.
model.tokenizer.apply_chat_template = partial(model.tokenizer.apply_chat_template, enable_thinking=False)
model.response_prefix = ''
placements = getattr(model.model, 'hf_device_map', {})
assert not any(str(device) in ('cpu', 'disk') for device in placements.values()), placements
manifest['device_map'] = {k: str(v) for k, v in placements.items()}
manifest['parameter_dtypes'] = dict(Counter(str(p.dtype) for p in model.model.parameters()))
components = Counter()
for layer in range(len(model.get_layers())):
    for name, modules in model.get_layer_modules(layer).items():
        components[name] += len(modules)
assert components == {'attn.o_proj': 32, 'mlp.down_proj': 32}, components
print('Target modules:', dict(components))
smoke = [Prompt(system=settings.system_prompt, user='What is 1+1?')]
print('Smoke response:', model.get_responses(smoke, skip_special_tokens=True)[0])
model.get_logprobs(smoke)  # Finite-logit check before the full run.
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
# 4. Load the same dataset slices as upstream defaults, with recorded revisions.
prompt_sets = {}
for label, dataset_id in [('good', 'mlabonne/harmless_alpaca'), ('bad', 'mlabonne/harmful_behaviors')]:
    data = load_dataset(dataset_id, revision=PROVENANCE['datasets'][dataset_id])
    for split, count in [('train', 400), ('test', 100)]:
        texts = list(data[split]['text'][:count])
        assert len(texts) == count
        prompt_sets[f'{label}_{split}'] = [Prompt(system=settings.system_prompt, user=t) for t in texts]
with (OUT / 'prompts.jsonl').open('w') as f:
    for split, prompts in prompt_sets.items():
        for index, prompt in enumerate(prompts):
            f.write(json.dumps({'split': split, 'index': index, 'system': prompt.system,
                                'prompt': prompt.user}) + '\n')
print({key: len(value) for key, value in prompt_sets.items()})

# Reuse the upstream scorer verbatim, while saving every generated answer.
judge = object.__new__(Evaluator)
judge.settings = settings

def first_token_logprobs(prompts, label):
    return torch.cat([model.get_logprobs(prompts[i:i+BATCH_SIZE])
                      for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc=label)])

@torch.inference_mode()
def evaluate_refusals(condition):
    rows = []
    with (OUT / f'{condition}_responses.jsonl').open('w') as f:
        for start in tqdm(range(0, 100, BATCH_SIZE), desc=condition):
            batch = prompt_sets['bad_test'][start:start+BATCH_SIZE]
            inputs, output_ids = model.generate(batch, max_new_tokens=MAX_NEW_TOKENS)
            sequences = output_ids[:, inputs['input_ids'].shape[1]:].detach().cpu().tolist()
            eos = model.model.generation_config.eos_token_id
            eos = set(eos if isinstance(eos, list) else [eos])
            for offset, (prompt, ids) in enumerate(zip(batch, sequences)):
                terminal = next((i for i, token in enumerate(ids) if token in eos), None)
                tokens = ids if terminal is None else ids[:terminal + 1]
                response = model.tokenizer.decode(tokens, skip_special_tokens=True)
                row = {'condition': condition, 'index': start + offset, 'prompt': prompt.user,
                       'response': response, 'refusal_marker_match': judge.is_refusal(response),
                       'generated_tokens': len(tokens), 'hit_token_limit': terminal is None}
                rows.append(row)
                f.write(json.dumps(row) + '\n')
                f.flush()
            if len(rows) % 10 == 0:
                print(condition, len(rows), '/ 100', flush=True)
    return {'refusals': sum(r['refusal_marker_match'] for r in rows), 'n': len(rows),
            'hit_token_limit': sum(r['hit_token_limit'] for r in rows)}


inputs_from_replay = list(Path('/kaggle/input').glob('**/baseline_logprobs.pt'))
assert len(inputs_from_replay)==1, 'Attach completed Phase 1 kernel output.'
replay_dir = inputs_from_replay[0].parent
baseline_logprobs = torch.load(inputs_from_replay[0], map_location='cpu', weights_only=True)
replay_summary = json.loads((replay_dir / 'summary.json').read_text())
# Confirm exact prompt provenance before reusing the stored distributions.
assert (OUT / 'prompts.jsonl').read_bytes() == (replay_dir / 'prompts.jsonl').read_bytes()
reference_logprobs = first_token_logprobs(prompt_sets['good_test'], 'Published checkpoint KL')
reference_kl = F.kl_div(reference_logprobs, baseline_logprobs, reduction='batchmean', log_target=True).item()
assert np.isfinite(reference_kl)
reference_metrics = evaluate_refusals('published_fp16')
diagnostic = {'condition':'published_checkpoint_fp16', 'model': PROVENANCE['reference_model'],
              'baseline': replay_summary['baseline'], 'rounded_replay': replay_summary['edited'],
              'published_checkpoint_measured':{**reference_metrics,'benign_first_token_kl':reference_kl},
              'published_card': PROVENANCE['published_metrics'],
              'scope':'100-prompt marker scorer, 100-token greedy non-thinking FP16; not HarmBench'}
(OUT / 'published_checkpoint_diagnostic.json').write_text(json.dumps(diagnostic,indent=2))
manifest['status']='evaluated'
(OUT / 'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(diagnostic,indent=2))


In [ ]:
from huggingface_hub import HfApi
api=HfApi()
repo_id=api.whoami()['name']+'/Qwen3.5-9B-heretic-replay-20261107'
api.upload_folder(repo_id=repo_id,folder_path=str(OUT),path_in_repo='diagnostics/published_fp16',
                  commit_message='Evaluate actual pinned published checkpoint under replay protocol')
print('Diagnostic persisted:', 'https://huggingface.co/'+repo_id)
